In [1]:
import sys
sys.path.append('/home/jovyan/work/tactics_demo/tools')
import os
import glob

In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
sys.path.append('/home/jovyan/base_demo')
import base_tool

In [5]:
import pandas as pd
import numpy as np
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

In [6]:
import numpy as np
from typing import List, Dict, Any, Optional

class FeatureExtractor:
    def __init__(self, snap_slice: List[Dict[str, Any]]):
        if not snap_slice:
            raise ValueError("snap_slice cannot be empty")
        self.snap_slice = list(snap_slice)
        self.last = snap_slice[-1]
        self.bid_book = self.last.get('bid_book', [])
        self.ask_book = self.last.get('ask_book', [])

        self.bid_volume = sum(vol for row in self.snap_slice for _, vol in row['buy_trade'])
        self.ask_volume = sum(vol for row in self.snap_slice for _, vol in row['sell_trade'])
        self.bid_volume_60 = sum(vol for row in self.snap_slice[-60:] for _, vol in row['buy_trade'])
        self.ask_volume_60 = sum(vol for row in self.snap_slice[-60:] for _, vol in row['sell_trade'])

    @staticmethod
    def _safe_get_level(book: List[tuple], idx: int = 0) -> tuple:
        if book and len(book) > idx:
            return book[idx]
        return (np.nan, 0)
    
    @property
    def price_last(self) -> float:
        return self.last.get('price_last', np.nan)
    
    @property
    def best_bid(self) -> float:
        return self._safe_get_level(self.bid_book)[0]
    
    @property
    def best_ask(self) -> float:
        return self._safe_get_level(self.ask_book)[0]
    
    @property
    def spread(self) -> float:
        bid, ask = self.best_bid, self.best_ask
        if np.isnan(bid) or np.isnan(ask):
            return np.nan
        return ask - bid
    
    @property
    def volatility(self) -> float:
        prices = [snap['price_last'] for snap in self.snap_slice 
                  if snap.get('price_last') is not None]
        if len(prices) < 2:
            return 0.0
        mean_price = np.mean(prices)
        if mean_price == 0:
            return 0.0
        return np.std(prices) / mean_price
    
    
    @property
    def wamp(self) -> float:
        bid_price, bid_vol = self._safe_get_level(self.bid_book)
        ask_price, ask_vol = self._safe_get_level(self.ask_book)
        numerator = bid_price * bid_vol + ask_price * ask_vol
        denominator = bid_vol + ask_vol
        if denominator == 0 or np.isnan(numerator):
            return 0.0
        return numerator / denominator
        
    @property
    def alpha_01(self) -> float:
        return self.bid_volume_60 / self.bid_volume if self.bid_volume > 0 else 0.0

    @property
    def alpha_02(self) -> float:
        return self.ask_volume_60 / self.ask_volume if self.ask_volume > 0 else 0.0
    
    @property
    def alpha_03(self) ->float:
        return (self.bid_volume_60 - self.ask_volume_60) / (self.bid_volume + self.ask_volume) if (self.bid_volume + self.ask_volume) > 0 else 0.0
    
    @property
    def alpha_04(self) -> float:
        num = self.snap_slice[-60]['num_trades'] - self.snap_slice[-1]['num_trades']
        return num / 60 if num > 0 else 0.0
    
    
    def extract_all(self) -> Dict[str, Any]:
        return {
            'price_last': self.price_last,
            'num_trades': self.last.get('num_trades', 0),
            'best_bid': self.best_bid,
            'best_ask': self.best_ask,
            'volatility': self.volatility,
            'spread': self.spread,
            'WAMP': self.wamp,
            'alpha_01': self.alpha_01,
            'alpha_02': self.alpha_02,
            'alpha_03': self.alpha_03,
            'alpha_04': self.alpha_04   
        }

def create_feature(snap_slice: List[Dict[str, Any]]) -> Dict[str, Any]:
    return FeatureExtractor(snap_slice).extract_all()

In [7]:
import numpy as np
import pandas as pd

class TrainValidTest():
    def __init__(self, snap_list, param_dict,feature_func,y_func):
        if param_dict is not None:
            self.__dict__.update(param_dict)
        # 确保必要属性存在
        if not hasattr(self, 'x_window'):
            self.x_window = 1
        if not hasattr(self, 'y_window'):
            self.y_window = 1

        self.snap_list = snap_list.copy()
        self.create_feature = feature_func
        self.create_y = y_func

    def samples(self):
        feature_records = []   # 存放特征字典
        labels = []            # 存放标签（标量）
        n = len(self.snap_list)
        stride = getattr(self, 'stride', 1)

        for i in range(self.x_window, n - self.y_window, stride):


            x_dict = self.create_feature(self.snap_list[i - self.x_window:i])
            # 直接从字典获取 volatility，避免 .iloc
            volatility = x_dict.get('volatility', 0.0)
            y_val = self.create_y(
                self.snap_list[i:i + self.y_window],
                volatility, self.k_up, self.k_down
            )
            feature_records.append(x_dict)
            labels.append(y_val)

        if not feature_records:
            return pd.DataFrame(), pd.Series(dtype=float)

        X_all = pd.DataFrame(feature_records)
        y_all = pd.Series(labels)
        return X_all, y_all

def samples_from_dates(dates, instrument_id, param_dict, create_feature, create_y):
    X_all_list = []
    y_all_list = []
    
    for date in dates:
        try:
            snap_list = base_tool.snap_list_load(instrument_id, date)
            if len(snap_list) < param_dict['x_window'] + param_dict['y_window']:
                print(f"{date}: 数据不足，跳过")
                continue
            tv = TrainValidTest(snap_list, param_dict, create_feature, create_y)
            X_day, y_day = tv.samples()
            X_all_list.append(X_day)
            y_all_list.append(y_day)
            print(f"{date}: 产生 {len(X_day)} 个样本")
        except Exception as e:
            print(f"{date}: 加载失败 - {e}")
    
    if X_all_list:
        X_total = pd.concat(X_all_list, axis=0, ignore_index=True)
        y_total = pd.concat(y_all_list, axis=0, ignore_index=True)
    else:
        X_total = pd.DataFrame()
        y_total = pd.Series()
    
    return X_total, y_total

def create_y(snap_slice, volatility, k_up, k_down):
    # 初始化突破时间索引为 None
    t_up = None
    t_down = None

    start = snap_slice[0]['price_last']
    if start is None or start == 0 or pd.isna(start):
        return pd.Series([0])

    up = start * (1 + volatility * k_up)
    down = start * (1 - volatility * k_down)

    for i in range(1, len(snap_slice)):
        price = snap_slice[i]['price_last']
        if price is None or pd.isna(price):
            continue

        if t_up is None and price >= up:
            t_up = i
        if t_down is None and price <= down:
            t_down = i

        if t_up is not None and t_down is not None:
            break

    # 根据触发情况决定标签
    if t_up is not None and t_down is not None:
        label = 1 if t_up < t_down else -1
    elif t_up is not None:
        label = 1
    elif t_down is not None:
        label = -1
    else:
        label = 0

    return label



## XGBoost 模型训练

In [8]:
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report

instrument_id = '518880'
train_days = 35
valid_days = 9
test_days = 10

trade_dates = ["20260105"
,"20260106"
,"20260107"
,"20260108"
,"20260109"
,"20260112"
,"20260113"
,"20260114"
,"20260115"
,"20260116"
,"20260119"
,"20260120"
,"20260121"
,"20260122"
,"20260123"
,"20260126"
,"20260127"
,"20260128"
,"20260129"
,"20260130"
,"20260202"
,"20260203"
,"20260204"
,"20260205"
,"20260206"
,"20260209"
,"20260210"
,"20260211"
,"20260212"
,"20260213"
,"20260224"
,"20260225"
,"20260226"
,"20260227"
,"20260302"
,"20260303"
,"20260304"
,"20260305"
,"20260306"
,"20260309"
,"20260310"
,"20260311"
,"20260312"
,"20260313"
,"20260316"
,"20260317"
,"20260318"
,"20260319"
,"20260320"
,"20260323"
,"20260324"
,"20260325"
,"20260326"
,"20260327"]
print(f"总交易日数量: {len(trade_dates)}")
print(f"交易日范围: {trade_dates[0]} ~ {trade_dates[-1]}")

总交易日数量: 54
交易日范围: 20260105 ~ 20260327


In [9]:
train_dates = trade_dates[:train_days]
valid_dates = trade_dates[train_days:train_days + valid_days]
test_dates = trade_dates[train_days + valid_days:train_days + valid_days + test_days]

print(f"训练集: {train_dates[0]} ~ {train_dates[-1]} ({len(train_dates)}天)")
print(f"验证集: {valid_dates[0]} ~ {valid_dates[-1]} ({len(valid_dates)}天)")
print(f"测试集: {test_dates[0]} ~ {test_dates[-1]} ({len(test_dates)}天)")

训练集: 20260105 ~ 20260302 (35天)
验证集: 20260303 ~ 20260313 (9天)
测试集: 20260316 ~ 20260327 (10天)


In [10]:
%%time
instrument_id = '511520'
trade_ymd = '20260319'
param_dict = {

    'name' : 'TBM',
    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,
    
    'x_window' : 300 ,
    'y_window' : 300 ,
    'stride': 60,

    'k_up': 3,
    'k_down': 3
}

print("生成训练集样本...")
X_train, y_train = samples_from_dates(train_dates, instrument_id, param_dict, create_feature, create_y)
print(f"训练集样本: X={X_train.shape}, y={y_train.shape}")
if len(y_train) > 0:
    print(f"标签分布:\n{y_train.value_counts()}")

生成训练集样本...
20260105: 产生 231 个样本
20260106: 产生 231 个样本
20260107: 产生 231 个样本
20260108: 产生 231 个样本
20260109: 产生 231 个样本
20260112: 产生 231 个样本
20260113: 产生 231 个样本
20260114: 产生 231 个样本
20260115: 产生 231 个样本
20260116: 产生 231 个样本
20260119: 产生 231 个样本
20260120: 产生 231 个样本
20260121: 产生 231 个样本
20260122: 产生 231 个样本
20260123: 产生 231 个样本
20260126: 产生 231 个样本
20260127: 产生 231 个样本
20260128: 产生 231 个样本
20260129: 产生 231 个样本
20260130: 产生 231 个样本
20260202: 产生 231 个样本
20260203: 产生 231 个样本
20260204: 产生 231 个样本
20260205: 产生 231 个样本
20260206: 产生 231 个样本
20260209: 产生 231 个样本
20260210: 产生 231 个样本
20260211: 产生 231 个样本
20260212: 产生 231 个样本
20260213: 产生 231 个样本
20260224: 产生 231 个样本
20260225: 产生 231 个样本
20260226: 产生 231 个样本
20260227: 产生 231 个样本
20260302: 产生 231 个样本
训练集样本: X=(8085, 11), y=(8085,)
标签分布:
 0    3848
 1    2207
-1    2030
Name: count, dtype: int64
CPU times: user 7.38 s, sys: 93.9 ms, total: 7.47 s
Wall time: 7.47 s


In [11]:
%%time
print("生成验证集样本...")
X_valid, y_valid = samples_from_dates(valid_dates, instrument_id, param_dict, create_feature, create_y)
print(f"验证集样本: X={X_valid.shape}, y={y_valid.shape}")
if len(y_valid) > 0:
    print(f"标签分布:\n{y_valid.value_counts()}")

生成验证集样本...
20260303: 产生 231 个样本
20260304: 产生 231 个样本
20260305: 产生 231 个样本
20260306: 产生 231 个样本
20260309: 产生 231 个样本
20260310: 产生 231 个样本
20260311: 产生 231 个样本
20260312: 产生 231 个样本
20260313: 产生 231 个样本
验证集样本: X=(2079, 11), y=(2079,)
标签分布:
 0    1001
 1     583
-1     495
Name: count, dtype: int64
CPU times: user 1.74 s, sys: 39.1 ms, total: 1.78 s
Wall time: 1.78 s


In [12]:
from sklearn.preprocessing import LabelEncoder

y_train_flat = y_train.values.ravel() if hasattr(y_train, 'values') else y_train.ravel()
y_valid_flat = y_valid.values.ravel() if hasattr(y_valid, 'values') else y_valid.ravel()

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train_flat)
y_valid_encoded = le.transform(y_valid_flat)
print(f"标签编码: {le.classes_} -> [0, 1, 2]")
print(f"编码后训练集标签分布: {pd.Series(y_train_encoded).value_counts().sort_index().to_dict()}")

标签编码: [-1  0  1] -> [0, 1, 2]
编码后训练集标签分布: {0: 2030, 1: 3848, 2: 2207}


In [14]:
%%time
model = xgb.XGBClassifier(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.003,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',
    num_class=3,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

model.fit(
    X_train, y_train_encoded,
    eval_set=[(X_valid, y_valid_encoded)],
    verbose=100
)

[0]	validation_0-mlogloss:1.05016
[100]	validation_0-mlogloss:1.02257
[200]	validation_0-mlogloss:1.00520


[300]	validation_0-mlogloss:0.99360
[400]	validation_0-mlogloss:0.98472
[500]	validation_0-mlogloss:0.97848
[600]	validation_0-mlogloss:0.97415
[700]	validation_0-mlogloss:0.97214
[800]	validation_0-mlogloss:0.97142
[900]	validation_0-mlogloss:0.97177
[999]	validation_0-mlogloss:0.97288
CPU times: user 2.22 s, sys: 39.5 ms, total: 2.26 s
Wall time: 590 ms


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softmax'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fro

In [15]:
y_pred_encoded = model.predict(X_valid)
y_pred = le.inverse_transform(y_pred_encoded)
print("验证集准确率:", accuracy_score(y_valid, y_pred))

unique_labels = sorted(set(y_valid.unique()) | set(np.unique(y_pred)))
label_names = { -1: '下跌(-1)', 0: '持平(0)', 1: '上涨(1)' }
target_names = [label_names.get(l, str(l)) for l in unique_labels]
print("分类报告:")
print(classification_report(y_valid, y_pred, labels=unique_labels, target_names=target_names))

验证集准确率: 0.5363155363155363
分类报告:
              precision    recall  f1-score   support

      下跌(-1)       0.39      0.34      0.37       495
       持平(0)       0.58      0.89      0.70      1001
       上涨(1)       0.48      0.09      0.15       583

    accuracy                           0.54      2079
   macro avg       0.48      0.44      0.41      2079
weighted avg       0.51      0.54      0.47      2079



## 测试集预测

In [16]:
%%time
print("生成测试集样本...")
X_test, y_test = samples_from_dates(test_dates, instrument_id, param_dict, create_feature, create_y)
print(f"测试集样本: X={X_test.shape}, y={y_test.shape}")

生成测试集样本...
20260316: 产生 231 个样本
20260317: 产生 231 个样本
20260318: 产生 231 个样本
20260319: 产生 231 个样本
20260320: 产生 231 个样本
20260323: 产生 231 个样本
20260324: 产生 231 个样本
20260325: 产生 231 个样本
20260326: 产生 231 个样本
20260327: 产生 231 个样本
测试集样本: X=(2310, 11), y=(2310,)
CPU times: user 2.39 s, sys: 43.1 ms, total: 2.43 s
Wall time: 2.51 s


In [17]:
y_test_flat = y_test.values.ravel() if hasattr(y_test, 'values') else y_test.ravel()
y_test_encoded = le.transform(y_test_flat)
y_test_pred_encoded = model.predict(X_test)
y_test_pred = le.inverse_transform(y_test_pred_encoded)
print("测试集准确率:", accuracy_score(y_test_flat, y_test_pred))
print("\n分类报告:")
print(classification_report(y_test_flat, y_test_pred, target_names=['下跌(-1)', '持平(0)', '上涨(1)']))

测试集准确率: 0.5251082251082251

分类报告:
              precision    recall  f1-score   support

      下跌(-1)       0.36      0.55      0.43       568
       持平(0)       0.63      0.80      0.70      1107
       上涨(1)       0.65      0.03      0.05       635

    accuracy                           0.53      2310
   macro avg       0.55      0.46      0.40      2310
weighted avg       0.57      0.53      0.46      2310



## 基于模型的回测

In [ ]:
class StrategyDemo():
    def __init__(self, model, param_dict={}) -> None:
        self.__dict__.update(param_dict)
        self.model = model
        self.feature_buffer = []
        self.position_last = 0
        self.prev_signal = 0
        
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}_xgb.pkl'
        if os.path.exists(data_file):
            os.remove(data_file)

    def on_snap(self, snap:dict) -> None:
        price = snap['price_last']
        
        if price == 0.0 or price == None:
            return
        
        self.feature_buffer.append(snap)
        
        if len(self.feature_buffer) < self.x_window:
            self.position_last = 0
            self.prev_signal = 0
            return
        
        if len(self.feature_buffer) > self.x_window:
            self.feature_buffer.pop(0)
        
        features_dict = create_feature(self.feature_buffer[-self.x_window:])
        X_pred = pd.DataFrame([features_dict])

        confidence = np.max(self.model.predict_proba(X_pred)[0])
        prediction = self.model.predict(X_pred)[0]

        if confidence > 0.5 and prediction != self.prev_signal:
            self.position_last = int(prediction)
            self.prev_signal = int(prediction)

        return

In [19]:
import os
import sys
current_notebook_path = os.path.abspath('%pwd') 
current_dir = os.path.dirname(current_notebook_path)
parent_dir = os.path.dirname(current_dir)
utils_path = os.path.join(parent_dir, 'tools')
sys.path.append(utils_path)

In [23]:
%%time
from multi_day_backtest import backtest_multi_days
param_dict['trade_ymd'] = ''
strategy = StrategyDemo(model,param_dict)
backtest_df = backtest_multi_days(instrument_id,'20260303','20260329',StrategyDemo,model,param_dict)

日期 20260303 完成，盈亏: 4.90, 成交: 1次
日期 20260304 完成，盈亏: 10.60, 成交: 1次
日期 20260305 完成，盈亏: 2.60, 成交: 1次
CPU times: user 5min 30s, sys: 1.39 s, total: 5min 31s
Wall time: 1min 50s


KeyboardInterrupt: 

In [64]:
backtest_df

,trade_ymd,order_time,order_price,total,trade,cancel,hold,profit_last,profits,maxdd,MAR,pper,trade_date
0,20260303,2026-03-03 14:55:00,115.813,2,2,0,0,4.9,4.9,1.0,4.9,2.45,2026-03-03
1,20260306,2026-03-06 14:55:00,115.976,2,2,0,0,2.5,2.5,1.0,2.5,1.25,2026-03-06
2,20260309,2026-03-09 14:55:00,115.739,2,2,0,0,-9.8,-9.8,9.8,-1.0,-4.90,2026-03-09
3,20260310,2026-03-10 14:55:00,115.747,2,2,0,0,-3.8,-3.8,3.8,-1.0,-1.90,2026-03-10
4,20260311,2026-03-11 14:55:01,115.761,2,2,0,0,0.3,0.3,1.0,0.3,0.15,2026-03-11
5,20260312,2026-03-12 14:55:00,115.811,3,2,1,0,8.0,8.0,1.0,8.0,4.00,2026-03-12
6,20260313,2026-03-13 14:55:00,115.894,2,2,0,0,6.1,6.1,1.0,6.1,3.05,2026-03-13
7,20260316,2026-03-16 14:55:00,115.794,2,2,0,0,-2.8,-2.8,2.8,-1.0,-1.40,2026-03-16
8,20260317,2026-03-17 14:55:00,115.895,2,2,0,0,1.9,1.9,1.0,1.9,0.95,2026-03-17
9,20260318,2026-03-18 14:55:00,116.027,2,2,0,0,9.0,9.0,1.0,9.0,4.50,2026-03-18


In [65]:
from backtesting import backtest_summary
summary = backtest_summary(backtest_df)
print(summary)

{'交易天数': 12, '累计盈亏': np.float64(20.1), '最大单日盈利': 13.5, '最大单日亏损': -9.8, '盈利天数': 8, '亏损天数': 4, '平盘天数': 0, '胜率(%)': np.float64(66.67), '日均盈亏': np.float64(1.68), '盈亏比': np.float64(0.89)}
